In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [6]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'time '
                                                                          'works '
                    

In [7]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John, no problem at all. Let me know what time works best for you to reschedule our meeting. Best, Seán.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John, no problem at all. Let me know what time works best for you to reschedule our meeting. Best, Seán.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='f403e60de82a8959f48e8f35367334cb')]


In [8]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem at all. Let me know what time works best for you to reschedule our meeting. Best, Seán.


## Approve

In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='b407c30e-9ba2-489a-bbd3-0288c2411264'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "inbox"}'}, '__gemini_function_call_thought_signatures__': {'call_766424': 'EnMKcQFpFH0TPH1nRqHjw1zoL9nhnNU5ca91h+woVEiFHmdeLCwvvC+ks34IsFKzW5Exl/fEWjQeqiTyHLlGS6u9r1iO0LUvltWkUIvY/Bwzm4BHbZQ3fORGNX4IpQgM7EyZReTOHAcoI8Fki5GtOXnSXi2V'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a10186-d962-70e1-9832-4394692027db-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'inbox'}, 'id': 'call_766424', 'type': 'tool_call'}], invalid_

## Reject

In [10]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='b407c30e-9ba2-489a-bbd3-0288c2411264'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "inbox"}'}, '__gemini_function_call_thought_signatures__': {'call_766424': 'EnMKcQFpFH0TPH1nRqHjw1zoL9nhnNU5ca91h+woVEiFHmdeLCwvvC+ks34IsFKzW5Exl/fEWjQeqiTyHLlGS6u9r1iO0LUvltWkUIvY/Bwzm4BHbZQ3fORGNX4IpQgM7EyZReTOHAcoI8Fki5GtOXnSXi2V'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a10186-d962-70e1-9832-4394692027db-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'inbox'}, 'id': 'call_766424', 'type': 'tool_call'}], invalid_

In [13]:
interrupts = response.get("__interrupt__", [])
if interrupts:
    print(interrupts[0].value["action_requests"][0]["args"]["body"])
else:
    print("No pending approval request. The rejection has been processed.")

No pending approval request. The rejection has been processed.


## Edit

In [12]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='b407c30e-9ba2-489a-bbd3-0288c2411264'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "inbox"}'}, '__gemini_function_call_thought_signatures__': {'call_766424': 'EnMKcQFpFH0TPH1nRqHjw1zoL9nhnNU5ca91h+woVEiFHmdeLCwvvC+ks34IsFKzW5Exl/fEWjQeqiTyHLlGS6u9r1iO0LUvltWkUIvY/Bwzm4BHbZQ3fORGNX4IpQgM7EyZReTOHAcoI8Fki5GtOXnSXi2V'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a10186-d962-70e1-9832-4394692027db-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'inbox'}, 'id': 'call_766424', 'type': 'tool_call'}], invalid_